# 5a Data Pre-Processing Functions

There is a saying in data analysis, and computer science more broadly: ‘garbage in, garbage out’. Even with the most sophisticated and elegantly designed machine learning algorithm in the world, if the data put into it is messy and bad, nothing useful will come from the model generated. A model is only as good as the data it is trained with.
<br>
<br>Classification algorithms look for subtle differences between samples in order to distinguish different groups of data. But if differences between samples are due to external factors, rather than genuine sample differences, such algorithms will be misdirected.
<br>
<br>A machine learning algorithm does not know to ignore small differences/anomalies – how can it? At best, these will confuse the algorithm. At worst, they will be interpreted as important features to learn from. It is therefore important that we eliminate experimental variation in data as much as possible so that only genuine differences are represented.
<br>
<br>Consider the two overlayed spectra below:
![Picture1.png](attachment:Picture1.png)
<br>
<br>You can easily attribute the fuzzy areas between 1500 - 2200 and 3500 - 4000 wavenumbers as noise (possibly due to a poor background), but an algorithm cannot. This can potentially be removed by smoothing the data.
<br>
<br>Similarly there are lare variations in transmittance, with even the baselines being different. This is likely caused by different amounts of sample being used for acquisition of the spectra and can also be corrected, *via* normalisation, before passing to the algorithm.

In this notebook, you will write some functions to carry out some pre-processing of the DataFrame of an IR spectrum. you will write functions to:

1. **Interpolate (replace decimal wavenumber rows (`400.482642`) with integer wavenumbers (`400`)).**
2. **Normalise (set the area under all spectra to 1).**
3. **Narrow the wavenumber range of the spectra.**

You will test out our functions on an example DataFrame - `m-anisaldehyde_1.txt`, since you are familiar with it from Notebook 4.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df_1 = pd.read_csv("m-anisaldehyde_1.txt",
                 skiprows=4,
                 sep=r"\s+",
                 names=["% Transmittance"],
                 index_col=0)
df_1.head()

,% Transmittance
399.826377,92.012424
400.183684,94.958885
400.540991,97.443201
400.898298,97.642822
401.255605,95.774480


## 1. Interpolation

In [ ]:
index_list = list(df_1.index)

The indices (wavenumbers) recorded in the DataFrame are specified to a high degree of precision on export from the Shimadzu IR software. The exact wavenumbers may not be consistent across files, and are also cumbersome to handle due the need to type large strings of numbers to specify particular DataFrame rows.
<br>
<br>In the following steps, we will interpolate the experimental data to evaluate the expected values at whole number wavenumbers. This will make the data easier to handle and comparable across spectra once all acquired data has been transformed in the same way.
<br>
<br>An outline of the process is as follows:
1. **Create a DataFrame of empty entries in which the index for each row is an integer between the maximum and minimum recorded wavenumbers in the IR data.**
2. **Join this DataFrame onto the IR DataFrame and sort the data by index to position each empty entry at its ordered position within the spectral data.**
3. **Interpolate a value at each empty entry by taking the weighted value of the two neighbouring data points**.
4. **Remove any rows containing non-integer indices to leave the interpolated results.**

In [ ]:
minimum_index = round(min(index_list))
maximum_index = round(max(index_list))
print(minimum_index, maximum_index)

400 4000


In [ ]:
# use of np.append(): numpy.append(arr, values, axis=None)
# arr: The input array to which elements are appended.
# values: The values to be appended to arr.

list1 = list(range(minimum_index, maximum_index+1))
# print(list1)

list2 = []
for i in list1:
  list2 = np.append(list2, np.nan)
  print(list2)

Streaming output truncated to the last 5000 lines.
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan


In [ ]:
df_2 = pd.DataFrame(data=list2, index=list1, columns=["% Transmittance"])
df_2.head()

# notice: don't put [] around list1/2, like [list1]

,% Transmittance
400,NaN
401,NaN
402,NaN
403,NaN
404,NaN


In [ ]:
df_2 = pd.DataFrame(
    data=list2,
    index=list1,
    columns=[df_2.columns[0]]
)

df_2.head()

,% Transmittance
400,NaN
401,NaN
402,NaN
403,NaN
404,NaN


In [ ]:
df = pd.concat([df_1,df_2])

df.tail()

,% Transmittance
3996.0,NaN
3997.0,NaN
3998.0,NaN
3999.0,NaN
4000.0,NaN


In [ ]:
df = df.sort_index()
df.head()
df.tail()

,% Transmittance
3999.000000,NaN
3999.335695,99.706163
3999.693002,99.483278
4000.000000,NaN
4000.050309,99.437475


In [ ]:
df = df.interpolate()

df.head()

,% Transmittance
399.826377,92.012424
400.000000,93.485654
400.183684,94.958885
400.540991,97.443201
400.898298,97.642822


Finally, we want to remove any rows with non-integer indices.

In [ ]:
df = df.loc[df.index.isin(list1)]

df.head()

,% Transmittance
400.0,93.485654
401.0,96.708651
402.0,97.294286
403.0,93.727395
404.0,92.881254


In [ ]:
# my version:
def interpolation(df_1):
  index_list = list(df_1.index)
  minimum_index = round(min(index_list))
  maximum_index = round(max(index_list))
  list1 = list(range(minimum_index, maximum_index+1))
  list2 = []
  for i in list1:
    list2 = np.append(list2, np.nan)
  df_2 = pd.DataFrame(data=list2, index=list1, columns=["% Transmittance"])
  df_2 = pd.DataFrame(
    data=list2,
    index=list1,
    columns=[df_2.columns[0]]
)
  df = pd.concat([df_1,df_2])
  df = df.sort_index().interpolate().loc[df.index.isin(list1)]
  return df

print(df.head())
print(df.tail())

In [ ]:
# Ryan corrected:
def interpolation(df_1):
  index_list = list(df_1.index)
  minimum_index = round(min(index_list))
  maximum_index = round(max(index_list))
  list1 = list(range(minimum_index, maximum_index+1))
  list2 = []
  for i in list1:
    list2 = np.append(list2, np.nan)
  # df_2 = pd.DataFrame(data=list2, index=list1, columns=["% Transmittance"])
  df_2 = pd.DataFrame(
    data=list2,
    index=list1,
    columns=[df_1.columns[0]] # df_1 instead of df_2
)
  df = pd.concat([df_1,df_2])
  df = df.sort_index().interpolate().loc[df.index.isin(list1)]
  return df

print(df.head())
print(df.tail())

       % Transmittance
400.0        93.485654
401.0        96.708651
402.0        97.294286
403.0        93.727395
404.0        92.881254
        % Transmittance
3996.0        99.850405
3997.0        99.322068
3998.0        99.536503
3999.0        99.595892
4000.0        99.460376


In [ ]:
df_3 = pd.read_csv("m-anisaldehyde_1.txt",
                 skiprows=4,
                 sep=r"\s+",
                 names=["% Transmittance"],
                 index_col=0)

df_3 = interpolation(df_3)

print(df.head())
print(df.tail())

       % Transmittance
400.0        93.485654
401.0        96.708651
402.0        97.294286
403.0        93.727395
404.0        92.881254
        % Transmittance
3996.0        99.850405
3997.0        99.322068
3998.0        99.536503
3999.0        99.595892
4000.0        99.460376


## 2. Normalisation

The next function we need is one which normalises our DataFrame. By normalise, we mean set the area under the spectrum to have an integral of 1.

You should have written a function in Notebook 3 which takes in a list of x values, a list of y values, and returns the area under the curve they define.

In [ ]:
def area_under_curve():
    x_values = df.index
    y_values = df.values
    area = (x_values, y_values)
    area = 0.0
    for i in range(1, len(x_values)):
        area += 0.5 * (y_values[i] + y_values[i-1]) * (x_values[i] - x_values[i-1])
    return area

area = area_under_curve()
print(area)

[327544.5885085]


In [ ]:
col = df.columns[0]
df[col] = df[col]/327544.5885085
# have to define col first, because this is wrong:
# df.columns[0] = df.columns[0]/327544.5885085
print(df.head())
area = area_under_curve()
print(area)

       % Transmittance
400.0         0.000285
401.0         0.000295
402.0         0.000297
403.0         0.000286
404.0         0.000284
[1.]


In [ ]:
def normalisation(df):
  def area_under_curve():
    x_values = df.index
    y_values = df.values
    area = (x_values, y_values)
    area = 0.0
    for i in range(1, len(x_values)):
        area += 0.5 * (y_values[i] + y_values[i-1]) * (x_values[i] - x_values[i-1])
    return area
  area = area_under_curve()
  col = df.columns[0]
  df[col] = df[col]/327544.5885085
  return df

print(df.head())
area = area_under_curve()
print(area)

       % Transmittance
400.0         0.000285
401.0         0.000295
402.0         0.000297
403.0         0.000286
404.0         0.000284
[1.]


In [ ]:
df_3 = pd.read_csv("m-anisaldehyde_1.txt",
                 skiprows=4,
                 sep=r"\s+",
                 names=["% Transmittance"],
                 index_col=0)

df_3 = interpolation(df_3)
df_3 = normalisation(df_3)

print(df.head())
print(df.tail())

area = area_under_curve()
print(area)


       % Transmittance
400.0         0.000285
401.0         0.000295
402.0         0.000297
403.0         0.000286
404.0         0.000284
        % Transmittance
3996.0         0.000305
3997.0         0.000303
3998.0         0.000304
3999.0         0.000304
4000.0         0.000304
[1.]


## 3. Narrowing the Wavenumber Range

Recall that the ultimate goal of this exercise is to identify substitutions patterns from IR spectra. As introduced in the manual, this can chiefly, although not necessarily exclusively, be performed by considering the fingerprint region of the spectrum.
<br>
<br>The exact region which gives the best results has been optimised for this exercise and happens to be the range of $ 630 cm^{-1} - 880 cm^{-1}$.

In [ ]:
def narrow_range(df):
  df = df.loc[630:880]
  return df

df = narrow_range(df)

print(df.head())
print(df.tail())

       % Transmittance
630.0         0.000264
631.0         0.000265
632.0         0.000262
633.0         0.000258
634.0         0.000255
       % Transmittance
876.0         0.000238
877.0         0.000238
878.0         0.000237
879.0         0.000238
880.0         0.000239


In [ ]:
df_3 = pd.read_csv("m-anisaldehyde_1.txt",
                 skiprows=4,
                 sep=r"\s+",
                 names=["% Transmittance"],
                 index_col=0)

df_3 = interpolation(df_3)
df_3 = normalisation(df_3)
df_3 = narrow_range(df_3)

print(df.head())
print(df.tail())

       % Transmittance
630.0         0.000264
631.0         0.000265
632.0         0.000262
633.0         0.000258
634.0         0.000255
       % Transmittance
876.0         0.000238
877.0         0.000238
878.0         0.000237
879.0         0.000238
880.0         0.000239


## 4. Combining the Processes

In [ ]:
df_3 = pd.read_csv("m-anisaldehyde_1.txt",
                 skiprows=4,
                 sep=r"\s+",
                 names=["% Transmittance"],
                 index_col=0)

df_3 = interpolation(df_3)
df_3 = normalisation(df_3)
df_3 = narrow_range(df_3)

print(df.head())
print(df.tail())

       % Transmittance
630.0         0.000264
631.0         0.000265
632.0         0.000262
633.0         0.000258
634.0         0.000255
       % Transmittance
876.0         0.000238
877.0         0.000238
878.0         0.000237
879.0         0.000238
880.0         0.000239


---

## 5. Constructing a Library

It is not only professional programmers who can make libraries - anyone can.
<br>
<br>A library is nothing more than a Python file (`.py`), containing a collection of objects and functions. This file can be imported into other files, which can then use its functions and objects.
<br>
<br>A library is made and used in the following way:
 - Make a new .py file (create a .txt file and alter the file type suffix)
 - Import any other libraries your code needs (e.g. pandas, etc.)
 - Define one or more functions (`def`...)
 - In another file *or Jupyter Notebook* in the same folder, `import` the file
 - Use a *function* in the *library* with `library.function()`

In [ ]:
import C317

In [ ]:
df_4 = pd.read_csv("m-anisaldehyde_1.txt",
                 skiprows=4,
                 sep=r"\s+",
                 names=["% Transmittance"],
                 index_col=0)

df_4 = C317.interpolation(df_4)
df_4 = C317.normalisation(df_4)
df_4 = C317.narrow_range(df_4)

print(df.head())
print(df.tail())

       % Transmittance
630.0         0.000264
631.0         0.000265
632.0         0.000262
633.0         0.000258
634.0         0.000255
       % Transmittance
876.0         0.000238
877.0         0.000238
878.0         0.000237
879.0         0.000238
880.0         0.000239


---